# Historical Photo Monoplotting

'Monoplotting' is photogrammetric technique that reprojects a single oblique photograph into spatial data using georeferenced points in the image. This workflow is designed to map past glacier extent from individual historical photographs, but can be used for other applications that require accurate positions of a camera or feature in a photograph. The work here is inspired by the tools of Bozzini et al 2012 (International Journal of Heritage in the Digital Era) and Golparvar and Wang 2025 (Computers and Geosciences). 

The pipeline has two main sections: First, **resolve the 'camera model'** - that is, to calculate the camera sensor and lens parameters ('intrinsic parameters'), as well as camera view direction/orientation and position in the world ('extrinsic parameters'). Once the camera model is calculated, then you can **orthorectify the image** or draw **line features** on your image and accurately **project them back into the world space.**

To create the instrinsic camera model, ideally you know the lens focal length (mm) and camera sensor or film dimensions (mm). If not, there is function here to estimate the field of view of your image and therefore be able to continue reprojecting your image. To estimate the extrinsic camera model (and estimate an unknown focal legnth, sensor size etc) you require at least 6 'ground control points' (GCPs) of known latitude (northing), longitude (easting), and elevation, ideally spread evenly across your image width and height (and even depth). Together, this information is enough to position your camera in world space and its field of view. Once solved, rays can be cast out from the camera position to intersect with a digital elevation model of the landscape to locate any feature in the photo on a real world terrain. 

**Essential inputs** are the photo itself, ground control point coordinates and pixel positions, a digital elevation model (DEM) of the landscape in the photo, and any line features wanting to be projected. Ground control point XYZ coordinates can be visualy matched to pixel coordinates (row/col or UV) using high resolution DEMs, satellite, or georeferenced aerial imagery. Pixel coordinates can be extracted from various image software - we recommened using ImageJ which is free and open source. Optionally, masks to exlude certain features/areas in the photo can be included. See below for input data formats.

**The output** is primarily georectified line feature shapefiles with uncertainty quantification, but also an orthorectified image and camera position coordinates. 

This workflow was designed by Dr Ryan North and Elena Disilvestro, Monash University, with code assistance from Claude Opus 4.8 and Gemini 3.1 Pro.

---

## Input files

| File | Format | Notes |
|---|---|---|
| Oblique photo | JPG / PNG / TIFF | Colour or greyscale |
| DEM | GeoTIFF | **Projected CRS in metres** (e.g. UTM). Must cover everything visible in the photo |
| GCP table | CSV | `GCP, row, col, X, Y, Z` — at least 6 points, 8+ recommended |
| Line features | CSV | `feature_name, col, row` — only needed for Part 2 |
| Custom exclusion mask | Image | *Optional*. Image-space mask for lens flare, dust, aircraft struts… |
| Stable terrain mask | GeoTIFF or shapefile | *Optional*. Only for the warp-fill in Cell 15 |

### GCP CSV

```
GCP,row,col,X,Y,Z
GCP01,876,1523,355000.0,5820000.0,1450.0
GCP02,912,2104,355480.0,5819760.0,1512.0
```

### Line feature CSV

```
feature_name,col,row
terminus_1947,1523,876
terminus_1947,1687,901
moraine_outer,2104,912
```

The header row is **required** and the names must match exactly (surrounding
spaces are ignored, capitalisation is not). Columns are matched by name, so the
order they appear in the file does not matter. Rows are grouped by
`feature_name` and kept in file order, so keep each feature's vertices together
and in sequence.

---

## Conventions used throughout

- **`row`** = pixel row counted from the **top** of the image;
  **`col`** = pixel column counted from the **left**.
- OpenCV works in `(x, y) = (col, row)`, so GCP pixel arrays are stored as
  `[col, row]`. Swapping these two is by far the most common setup mistake —
  the plots in Cells 3, 6 and 10 exist to catch it.
- World coordinates are `(E, N, Z)` = `(Easting, Northing, Elevation)` in the
  **DEM's CRS, in metres**. A geographic (lat/lon) DEM will not work: reproject
  it to UTM or another metre-based CRS first.
- Every cell that has settings puts them in a `── Configuration ──` block at the
  top of the cell. Nothing else needs editing.

---

## Notebook outline

### Part 1 — Camera model and raycasting
| Cell | Purpose |
|---|---|
| 1 | Imports |
| 2 | File paths |
| 3 | Load photo and GCPs |
| 4 | Resolve camera field of view |
| 5 | Solve camera pose |
| 6 | Validate GCP reprojection |
| 7 | Load DEM |
| 8 | Build terrain mesh from DEM |
| 9 | Raycast — depth image and sky mask |

### Part 2 — Line feature projection
| Cell | Purpose |
|---|---|
| 10 | Load and validate digitised features |
| 11 | Project features to world coordinates |
| 12 | Write shapefile |
| 13 | Estimate line uncertainty |

### Part 3 — Orthorectification *(optional)*
| Cell | Purpose |
|---|---|
| 14 | Orthorectify stable terrain |
| 15 | Fill unstable terrain by warping |

Part 2 needs Cells 1–9 to have run in the same kernel session (it reuses the
camera model and the raycasting scene). Cells 14 and 15 are not prerequisites.

---

## Install

```
pip install numpy opencv-python rasterio open3d matplotlib scipy geopandas shapely
```

Note, Python 3.12 is required for Open3D to work.

## Cell 1 — Imports

Every processing step lives in `monoplotting_functions.py`, which must sit in
the same folder as this notebook. This cell imports it as `mf`.
`geopandas` and `open3d` are checked separately when it is imported, to avoid
long tracebacks in the case of an error.

In [ ]:
# Reload monoplotting_functions.py automatically if it is edited while the notebook is open
%load_ext autoreload
%autoreload 2

import os                                   # paths
import monoplotting_functions as mf         # every processing function used below

# ── Report versions so a broken environment is obvious immediately ────────────
mf.report_versions()

## Cell 2 — File Paths

Set paths for inputs and outputs here.

On Windows keep the `r"..."` prefix so backslashes are read literally.
Forward slashes work on every platform.

In [ ]:
# ── Required inputs ───────────────────────────────────────────────────────────
IMAGE_PATH = r"path/to/photo.jpg"          # the oblique photograph
DEM_PATH   = r"path/to/dem.tif"            # DEM GeoTIFF, projected CRS in metres
GCP_CSV    = r"path/to/gcps.csv"           # columns: GCP, row, col, X, Y, Z

# ── Outputs ───────────────────────────────────────────────────────────────────
# MESH_PATH is written once by Cell 8 and reused on later runs. Delete the file
# (or change the name) to force a rebuild with different mesh settings.
MESH_PATH   = r"path/to/dem_mesh.obj"      # terrain mesh, generated by Cell 8
OUTPUT_PATH = r"path/to/photo_ortho.tif"   # orthorectified GeoTIFF, Cell 14

# ── Part 2: line features ─────────────────────────────────────────────────────
FEATURES_CSV = r"path/to/lines.csv"        # columns: feature_name, col, row
# Shapefile is named after the ortho so outputs from one photo stay together.
OUTPUT_SHP   = os.path.splitext(OUTPUT_PATH)[0] + '_lines.shp'

# ── Optional: custom exclusion mask (used by Cell 9) ─────────────────────────
# A mask in IMAGE PIXEL space, same dimensions as the photo. Non-zero pixels are
# excluded from every projection step. Use it to mask things that are not in the terrain
# eg. lens flare, dust spots, an aircraft wing, 
# foreground vegetation, hazy distant ground you do not trust, etc.
# Set to None if not used.
CUSTOM_MASK_PATH = None

# ── Optional: stable terrain mask (used by Cells 14 and 15) ───────────────────
# Identifies which parts of the SCENE (3D dem mesh) are unchanged between the DEM and the photo,
# for example, areas of stable bedrock.
# Pixels inside the mask are orthorectified directly from the ray-mesh geometry.
# Pixels outside the mask (a glacier that has since thinned, a landslide, a changed
# shoreline) will be wrong in the modern DEM's geometry and are instead filled by the warp
# in Cell 15.
# Accepted: GeoTIFF (non-zero = stable) or a polygon shapefile of stable areas.
# Set to None to treat the whole scene as stable and skip Cell 15.
STABLE_MASK_PATH = None

# ── Report ────────────────────────────────────────────────────────────────────
mf.check_paths(IMAGE_PATH, DEM_PATH, GCP_CSV, MESH_PATH, OUTPUT_PATH,
               FEATURES_CSV, OUTPUT_SHP, CUSTOM_MASK_PATH, STABLE_MASK_PATH)

## Cell 3 — Load Photo and GCPs

Reads the photo and the GCP table, and draws the GCPs on the photo to check positions.

**Configuration:** `EXCLUDE_GCPS`

**Check the plot.** Every red cross should sit exactly on the feature it was
measured at. If the whole set looks mirrored or transposed, `row` and `col` are
probably swapped in the CSV. Come back here after Cell 6 to exclude any GCP with a large
reprojection error.

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
# Labels listed here are dropped before the pose solve. Leave empty on the first
# run; after Cell 6 reveals a bad point, add its label and re-run from this cell.
# Example: EXCLUDE_GCPS = ['GCP04', 'GCP07']
EXCLUDE_GCPS = []

# ── Load the photo ────────────────────────────────────────────────────────────
img_cv, img_rgb, h_img, w_img, aspect, n_bands = mf.load_photo(IMAGE_PATH)

# ── Load GCPs from CSV ────────────────────────────────────────────────────────
GCP_PIXEL, GCP_WORLD, gcp_labels = mf.load_gcps(GCP_CSV, w_img, h_img)
mf.print_gcp_table(gcp_labels, GCP_PIXEL, GCP_WORLD)

# ── Drop excluded GCPs ────────────────────────────────────────────────────────
GCP_PIXEL, GCP_WORLD, gcp_labels = mf.exclude_gcps(GCP_PIXEL, GCP_WORLD, gcp_labels, EXCLUDE_GCPS)

# ── Plot GCPs on the photo to visually check ──────────────────────────────────
mf.plot_gcps(img_rgb, gcp_labels, GCP_PIXEL)

## Cell 4 — Resolve Camera Field of View

To define the intrinsic camera model, the the camera's focal length in pixels is required.
The vertical field of view (FOV) can be used to calculate this. 
There are three ways to supply it, in order of preference:

**Option 1 — physical focal length + sensor size.** The most robust. Use the
*actual* focal length from the EXIF data or the lens barrel.
If sensor height is unknown it is derived from the image aspect ratio.

**Option 2 — state the vertical FOV directly.** If known from camera/lens setup.
Skips the need for calculating from focal length (mm) and sensor size.

**Option 3 — estimate it from the GCPs.** For unknown camera/lens setups.
`solvePnP` is run across a range of test FOVs and the value with the lowest
reprojection error wins. This only works if the GCPs constrain the geometry —
they need to span a wide elevation range (>300 m is a good target) and be spread
across the frame rather than clustered or linear. The code first runs a coarse estimate
testing 1 degree intervals, then a fine estimate at 0.1 degree intervals.

The plot shows reprojection error against FOV. A **sharp narrow minimum** means
the FOV is accurately calculated from the GCPs. A **broad flat trough** means many FOVs fit almost
equally well and the estimate should not be trusted — better to find the real camera specs.
The grey steps (right axis) show how many GCPs each FOV fits; they should peak at the best FOV.

**Outlier handling (Option 3).** A plain least-squares fit bends towards a badly matched
GCP (wrong feature, row/col swap, elevation from terrain that has since changed), and the
FOV absorbs the error first because it is the parameter the GCPs constrain least. So each
trial FOV is solved with RANSAC (as in Cell 5) and scored with a *capped* error: every GCP
contributes its squared error, but never more than `FOV_RANSAC_THRESHOLD`². A blunder
then adds the same fixed penalty at every FOV and cannot move the minimum, while a wrong
FOV is still charged for every GCP it fails to fit.

**Leave-one-out and FOV uncertainty (Option 3).** The search is repeated once per GCP
with that GCP removed (needs 7+ GCPs). The table shows each GCP's residual at the best FOV
and how far the FOV moves without it — an *outlier* has a large residual (the robust fit is
ignoring it), an *influential* GCP moves the FOV a long way on its own. Either way, check it
and add it to `EXCLUDE_GCPS` in Cell 3 if it is wrong. The spread of the leave-one-out
answers gives the jackknife standard error of the FOV, stored as `fov_sigma_deg` and carried
into the line uncertainty in Cell 13. For Options 1 and 2, set `FOV_UNCERTAINTY_DEG` instead
(0 = FOV known exactly).

Options 1 and 2 take priority if their values are known.

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
# Option 1 — physical focal length and sensor size (preferred).
# Use the actual physical focal length (distance in mm from lens to focal point, usually listed on lens)
# Use the physical size of the sensor (in mm). Usually recorded in camera specs. Standard '35mm' film is 24 x 36 mm)
FOCAL_LENGTH_MM  = None    # e.g. 18.0  mm
SENSOR_WIDTH_MM  = None    # e.g. 23.5  mm
SENSOR_HEIGHT_MM = None    # e.g. 15.6  mm, or None to derive from image aspect

# Option 2 — vertical FOV in degrees, used if Option 1 is not available.
VERTICAL_FOV_DEG = None    # e.g. 53.1

# Options 1 and 2 only: +/- uncertainty of the known FOV, in degrees. 0 treats it as
# exact. Option 3 works out its own uncertainty by leave-one-out (below).
FOV_UNCERTAINTY_DEG = 0.0

# Option 3 — estimate FOV from the GCPs. Only used if Options 1 and 2 are None.
ESTIMATE_FOV_FROM_GCPS = True

# Option 3 search range. Narrow it if you know roughly what the camera is
# (e.g. can be >100 deg for a wide angle lens, or as low at ~10 deg for a zoom lens)
FOV_SEARCH_MIN  = 10.0   # deg, lower bound of the search
FOV_SEARCH_MAX  = 120.0  # deg, upper bound of the search
FOV_COARSE_STEP = 1.0    # deg, coarse grid step
FOV_FINE_RANGE  = 3.0    # deg, fine search spans +/- this around the coarse best
FOV_FINE_STEP   = 0.1    # deg, fine grid step

# Option 3 outlier handling. Each trial FOV is solved with RANSAC and scored with a
# capped error, so a GCP further than this from its picked position (px) adds a fixed
# penalty and cannot drag the FOV towards itself. Scale it to image size and GCP
# quality. Cell 5 uses the same value by default, so both steps reject the same GCPs.
FOV_RANSAC_THRESHOLD = 20.0

# Option 3 leave-one-out test: repeat the search once per GCP with that GCP removed.
# Shows which GCPs the FOV depends on, and gives the FOV uncertainty used in Cell 13.
# Needs at least 7 GCPs. Each repeat searches +/- FOV_LOO_WINDOW deg around the estimate.
FOV_LEAVE_ONE_OUT = True
FOV_LOO_WINDOW    = 10.0   # deg

# ── Resolve the FOV (Option 1, then 2, then 3) ────────────────────────────────
# fov_sigma_deg is the FOV uncertainty (deg), carried into the line uncertainty in Cell 13.
fov, fov_search, fov_sigma_deg = mf.resolve_fov(
    GCP_PIXEL, GCP_WORLD, w_img, h_img, aspect,
    focal_length_mm=FOCAL_LENGTH_MM, sensor_width_mm=SENSOR_WIDTH_MM,
    sensor_height_mm=SENSOR_HEIGHT_MM, vertical_fov_deg=VERTICAL_FOV_DEG,
    fov_uncertainty_deg=FOV_UNCERTAINTY_DEG, estimate_from_gcps=ESTIMATE_FOV_FROM_GCPS,
    fov_search_min=FOV_SEARCH_MIN, fov_search_max=FOV_SEARCH_MAX,
    fov_coarse_step=FOV_COARSE_STEP, fov_fine_range=FOV_FINE_RANGE, fov_fine_step=FOV_FINE_STEP,
    fov_ransac_threshold=FOV_RANSAC_THRESHOLD, fov_leave_one_out=FOV_LEAVE_ONE_OUT,
    fov_loo_window=FOV_LOO_WINDOW, gcp_labels=gcp_labels)

# Plot the error curve (Option 3 only)
if fov_search is not None:
    mf.plot_fov_search(fov_search)

# ── Report and sanity check ───────────────────────────────────────────────────
focal_px = mf.report_fov(fov, h_img, fov_sigma_deg)

## Cell 5 — Solve Camera Pose

Finds the camera's position and orientation from the GCP correspondences using
`solvePnPRansac`.

**Configuration:** `RANSAC_THRESHOLD`

RANSAC repeatedly fits the pose to random subsets of GCPs and keeps the solution
that the most points agree with. Points that disagree by more than
`RANSAC_THRESHOLD` pixels are called outliers and excluded.

`RANSAC_THRESHOLD` defaults to Cell 4's `FOV_RANSAC_THRESHOLD` (20 px), so the FOV
search and the pose reject the same GCPs. RANSAC needs at least 5 GCPs that agree within
the threshold. If it cannot find them, the cell stops and lists how far each GCP is from
a plain fit, rather than quietly fitting every GCP (outliers included). A warning is
printed if the pose rests on fewer than 6 GCPs.

**Check:** look at the mean reprojection error. It is averaged over the *inlier* GCPs —
the ones RANSAC kept and the pose was solved from — and that is the value used for the
uncertainty downstream. The mean over all GCPs is printed alongside for transparency. What counts as 'good' depends on
image size and GCP quality — as a rough guide under ~5 px is very good, under ~20 px is workable,  
more than 20 px could probably be improved. Cell 6 shows which GCPs are contributing most to the error.

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
# RANSAC_THRESHOLD: reprojection error in pixels above which a GCP is treated as
# an outlier and dropped from the solve. Lower = stricter. Scale it to your image
# size and quality. RANSAC is 'random sample consensus' outlier detection method,
# kind of like 'leave one out'
# Defaults to Cell 4's FOV_RANSAC_THRESHOLD so the FOV search and the pose reject the
# same GCPs. If fewer than 5 GCPs agree within it, this cell stops and says so.
RANSAC_THRESHOLD = FOV_RANSAC_THRESHOLD     # or a number, e.g. 20.0

# ── Build the camera intrinsic matrix K ───────────────────────────────────────
K, focal_px, dist_coeffs = mf.build_camera_matrix(fov, w_img, h_img)

# ── Solve the pose and recover the camera position ────────────────────────────
# err holds every GCP's error; inlier_mask marks the GCPs RANSAC kept (the pose is
# solved from these, so downstream error and uncertainty use only these).
R, cam_world, uv, err, inlier_mask = mf.solve_pose(GCP_PIXEL, GCP_WORLD, gcp_labels, K, dist_coeffs,
                                                   RANSAC_THRESHOLD, fov, focal_px)

## Cell 6 — Validate GCP Reprojection

Draws each GCP twice: where it was picked, and where the solved camera model puts
it. The gap between the two is the reprojection error.

- **Red cross** — picked position
- **Blue circle** — model-projected position
- **Yellow line** — the error

What each pattern could mean:

- **All short lines in random directions** - great! Good GCPs.
- **One or two long lines, the rest short** — those GCPs are bad. Add them to
  `EXCLUDE_GCPS` in Cell 3 and re-run from there.
- **Errors growing towards the image corners** — uncorrected lens distortion.
  Add distortion coefficients in Cell 5.
- **Everything long and in a consistent direction** — the FOV is probably wrong.
  Revisit Cell 4.

In [ ]:
# Print table of picked vs projected GCPs.
mf.print_reprojection_table(gcp_labels, GCP_PIXEL, uv, err, inlier_mask)

# ── Plot picked vs projected, joined by an error line ─────────────────────────
mf.plot_reprojection(img_rgb, gcp_labels, GCP_PIXEL, uv, err, inlier_mask)

## Cell 7 — Load DEM

The DEM does three jobs: it supplies the terrain surface the photo is draped over,
it defines the CRS of every output, and its grid becomes the grid of the
orthorectified raster.

**Configuration:** `OUTPUT_RES_M`, `NODATA_VALUE`

**Check:** the CRS must be projected (metres) and the bounds must cover everything
visible in the photo, including distant ground near the horizon. Terrain outside
the DEM will not be orthorectified.

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
# OUTPUT_RES_M: pixel size of the output raster, in metres.
# None = keep the DEM's own resolution.
OUTPUT_RES_M = None

# NODATA_VALUE: value written where the photo gives no coverage (sky, terrain
# hidden behind a ridge, ground outside the frame). 0 renders as black. Set it
# transparent in QGIS/ArcGIS after loading.
NODATA_VALUE = 0

# ── Load DEM, define the output grid, and check the CRS ──────────────────────
# `dem` holds the elevation grid, its georeferencing and the output grid.
dem = mf.load_dem(DEM_PATH, OUTPUT_RES_M, cam_world=globals().get('cam_world'))

## Cell 8 — Build Terrain Mesh from DEM

Converts the DEM grid into a TIN mesh (triangular irregular network) (an '.obj' file) in real-world
coordinates. Raycasting (Cell 9) requires a mesh to 'hit' the world surface, 2D raster data would not work. 

This is the most time consuming step, so it preferentially runs once. If a file already exists at `MESH_PATH` (Cell 2 inputs) it is loaded instead of
rebuilt. To rebuild with different settings, delete that file or change the name
in Cell 2.

**Configuration:** `MESH_STRIDE`, `MESH_SIMPLIFY`, `SHOW_MESH_VIS`

The two size controls do different things:
`MESH_STRIDE` samples every X cells in the DEM to create the 
mesh. This is an easy way to reduce computtational  power required. However, too big of a stride will
blurs ridgelines which can matter in an oblique view because they define the skyline.
`MESH_SIMPLIFY` removes triangles *after* meshing using quadric decimation, which keeps triangles where the surface is
curved and removes them from flat ground. This is an effecient way to keep detail where it is needed. 
Prefer simplification over stride:
`MESH_STRIDE = 1` with `MESH_SIMPLIFY = 0.5` is a good starting point, and only
raise the stride if the DEM is too large to mesh at full resolution. May run into RAM or timeout problems here.

**Check:** the wireframe plot with its zoom inset shows triangle density. There
should be enough triangles to follow the terrain in the part of the scene you care
about.

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
# MESH_STRIDE: use every Nth DEM cell. 1 = full resolution. 2 = quarter of the
# vertices. Raise only if the DEM is too big to mesh at full resolution.
MESH_STRIDE = 1

# MESH_SIMPLIFY: fraction of triangles to keep after quadric decimation.
# 1.0 = no simplification, 0.5 = keep half (recommended starting point).
MESH_SIMPLIFY = 0.5                     # a fraction between 0 and 1

# SHOW_MESH_VIS: draw the wireframe check plot. Slow for very large meshes.
SHOW_MESH_VIS = True

# ── Build the mesh (or load it if MESH_PATH already exists) ───────────────────
mesh, verts, t = mf.build_or_load_mesh(MESH_PATH, dem, MESH_STRIDE, MESH_SIMPLIFY)

# ── Visual check: wireframe with a zoomed inset ───────────────────────────────
if SHOW_MESH_VIS:
    mf.plot_mesh(verts, t, MESH_STRIDE, MESH_SIMPLIFY)

## Cell 9 — Raycasting: Depth Image and Mask

Fires one ray from the camera through every image pixel and records where it hits
the terrain mesh. This is the step that ties pixels to ground.

Two arrays come out, both the same size as the photo:

- **`depth_image`** — distance along the ray to the terrain, in metres
  (0 where the ray hit nothing)
- **`exclusion_mask`** — True for pixels to ignore everywhere downstream

Sky detection is a useful by-product - a ray projected out without the mesh was likely pointed at the sky.
This geometrically detects the sky rather than using colour of a pixel for example, so it works effectively on white sky,
cloud, snow against sky, and low-contrast historical photographs where colour-based sky detection fails. This relies on a 
high-resolution terrain mesh.

**Configuration:** `MASK_MODE`

**Check:** depth image should look similar to real image.

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
# MASK_MODE — which pixels are excluded from every projection step:
#
#   'geometric'        Rays that miss the mesh are sky. Use this by default.
#
#   'geometric+custom' The above, plus CUSTOM_MASK_PATH from Cell 2. Use when
#                      the photo also contains non-terrain obstructions, eg. dust
#                      spots, lens flare, an aircraft wing, foreground clutter.
#
#   'custom_only'      Only CUSTOM_MASK_PATH; no geometric masking. Use when the
#                      DEM does not cover the whole photo and geometric masking
#                      would wrongly discard real terrain.
MASK_MODE = 'geometric'

# ── Build the raycasting scene ────────────────────────────────────────────────
scene = mf.build_raycasting_scene(MESH_PATH)

# ── Cast one ray per image pixel ──────────────────────────────────────────────
rays_world, depth_image, coverage, camera_lifted, dZ = mf.cast_image_rays(
    scene, K, R, cam_world, w_img, h_img)

# ── Show the depth image beside the photo ─────────────────────────────────────
mf.plot_depth_image(img_rgb, depth_image, coverage)

# ── Build the exclusion mask ──────────────────────────────────────────────────
exclusion_mask = mf.build_exclusion_mask(depth_image, MASK_MODE, CUSTOM_MASK_PATH, w_img, h_img)

---

# Part 2 — Line Feature Projection

Takes features digitised on the photograph in pixel coordinates and converts them
into real-world lines. Output is a shapefile in the DEM's coordinate reference system, with an uncertainty estimate
for each line feature.

**Requires Cells 1–9 to have run in this kernel session.** Part 2 reuses the
camera model from Cell 5 and the raycasting scene from Cell 9. Cells 14 and 15
are not needed.

### Digitising the features

Any tool that exports vertex coordinates works. Create a csv of line features. ImageJ/ Fiji is free and works: 

1. Open the photo, choose the **Segmented Line** tool, click along the feature.
2. Press `T` to add it to the ROI Manager, then **More ▸ List** to get the
   vertex table — ImageJ's `X` is the column, `Y` is the row.
3. Save as CSV, then edit the header line to read `feature_name,col,row` and add
   the feature's name in the first column — ImageJ's `X` is the column, `Y` is
   the row.

Repeat for each feature, appending to the same CSV. Rows are grouped by name and
kept in file order, so keep each feature's vertices together and in sequence.

### Digitise the right thing

Every vertex is turned into a world position by intersecting its ray with the
terrain surface. The answer is only correct where the thing you clicked on *is*
the terrain surface.

- **Glacier terminus** — digitise the line where ice meets the ground, not the
  top of the ice cliff. The top of a 30 m cliff projects to wherever the ray
  happens to strike the terrain behind it, which can be far beyond the true edge.
- **Moraines, trimlines, shorelines, bedrock features** — digitise on the ground
  surface itself.
- Anything standing above the terrain — vegetation, buildings, the ice cliff top
  — will be displaced, and the displacement grows with distance.

Any vertex whose ray misses the terrain (sky, or ground beyond the DEM) is
flagged and dropped.

## Cell 10 — Load and Validate Line Features

Reads the feature CSV and draws each feature on the photo.

**Check:** the lines should trace the features you digitised. If they are mirrored
or transposed, `col` and `row` are probably swapped in the CSV.

In [ ]:
# ── Read and validate the digitised features ──────────────────────────────────
features_valid = mf.load_features(FEATURES_CSV, w_img, h_img)

# ── Draw them on the photo ────────────────────────────────────────────────────
# `colours` is reused by Cell 12 so a feature keeps the same colour throughout.
colours = mf.plot_features(img_rgb, features_valid)

## Cell 11 — Project Features to World Coordinates

Casts a ray through every digitised vertex and takes the world position where it
meets the terrain mesh — the same operation Cell 9 did for whole-image pixels,
applied to a few hundred points.

Reusing the Cell 9 raycasting scene matters for two reasons: the BVH is already
built so this is near-instant, and the lines land on exactly the same surface as
the orthorectified image, so the two products agree with each other.

Vertices whose ray misses the mesh are reported and marked. Cell 12 splits the
line at those gaps rather than drawing a straight segment across them.

In [ ]:
# ── Project every feature onto the terrain ────────────────────────────────────
features_world = mf.project_features(features_valid, cam_world, R, K, scene,
                                     camera_lifted, dZ)

## Cell 12 — Write Shapefile

Writes the projected features to a shapefile in the DEM's CRS, then plots them
on the photo and on a DEM hillshade so the projection can be checked. 

Vertices that missed the mesh break a feature into segments: each run of
consecutive hits becomes a `LineString`, and a feature with more than one run is
stored as a `MultiLineString`. Nothing is interpolated across a gap.

**Attributes written:** `name`, `img_src`, `n_verts`, `n_hit`, `n_miss`,
`reproj_err` (mean over the inlier GCPs), `reproj_all` (mean over all GCPs).
Cell 13 adds the uncertainty columns.

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
# SHOW_FEATURE_PLOT: visualise/plot the shapefiles here?
# The shapefile is written either way. This only controls the figure.
SHOW_FEATURE_PLOT = True                     # set False to skip the plot

# ── Write the shapefile ───────────────────────────────────────────────────────
gdf = mf.write_feature_shapefile(features_world, OUTPUT_SHP, dem['out_crs'], IMAGE_PATH, err, inlier_mask)

# ── Check plot: photo beside DEM hillshade ────────────────────────────────────
if SHOW_FEATURE_PLOT:
    mf.plot_projected_features(img_rgb, features_world, colours, dem)

## Cell 13 — Line Uncertainty

Estimates how far each projected line could be from the truth, in metres, and
writes the result back into the shapefile.

**Configuration:** `DIGITISE_PX`, `INCLUDE_DEM_ERROR`, `DEM_VERTICAL_ERROR_M`

Uncertainty is computed per pixel along the line, not per digitised vertex, so
that a feature clicked with 5 vertices and one clicked with 200 get comparable
numbers. Up to four components combine in quadrature:

| Term | Source | Formula |
|---|---|---|
| `σ_camera` | camera model error | (mean reprojection error of the inlier GCPs / focal px) × range |
| `σ_digitise` | clicking accuracy | (`DIGITISE_PX` / focal px) × range |
| `σ_FOV` | FOV uncertainty (`fov_sigma_deg`, Cell 4) | ground shift when the camera is re-solved at FOV ± `fov_sigma_deg` |
| `σ_DEM` | DEM vertical error | vertical error × tan(view zenith angle) |

The first two scale with **range**: an angular error of one pixel spans more
ground the further away it is. A line that is approximately *'front on'* to the camera
has a similar uncertainty across its length and can b fairly represented by the mean uncertainty
of each of the pixels in the line. A line extending away from the camera
has uncertainty that grows steadily along its length. Here we calculate an uncertainty gradient (
`σ_grad`) and automatically label the line as *depth-varying* when the far end is more than twice as uncertain the
near end. It is up to your judgement whether the mean or range of error (min and max) better represents your uncertainty. 

`σ_DEM` is represents a vertical DEM error that displaces the intersection point
horizontally by an amount that depends on how obliquely the ray strikes the
surface. A ray coming in near-vertically is barely affected, but a near horizontal ray is
affected enormously.  

`σ_FOV` carries the FOV uncertainty from Cell 4 onto the ground. A different FOV changes
the whole camera solution — the pose solve moves the camera to keep fitting the GCPs — so
there is no simple formula. Instead the camera is re-solved at FOV − σ and FOV + σ, each
line is re-cast through both, and the RMS of how far each point moves is its `σ_FOV`.
Each re-cast ray is matched to where it crosses the terrain *nearest the original point*,
not the first surface it meets: a camera close to the ground can end up just below it, or
behind a small rise, when re-solved, and its rays would otherwise all stop beside it.
Features among the GCPs move little; features outside their spread (especially the near
foreground) move with the camera and get a larger `σ_FOV`.
It is zero when the FOV is treated as exact. Like `σ_camera`, it is a camera-model error, so
it is part of the relative uncertainty.

**Relative vs absolute.** With `INCLUDE_DEM_ERROR = False` you get *relative*
uncertainty, which treats the DEM as truth. This is the right number for
comparing features measured against the same DEM — a terminus in 1950 versus 2020,
say, where a shared DEM bias cancels out. Set `INCLUDE_DEM_ERROR = True` for *absolute* uncertainty
against real ground truth, where `σ_DEM` is used to calculate an absolute uncertainty. 

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
# DIGITISE_PX: how accurately the line was clicked, in pixels. 1 px is reasonable baseline.
DIGITISE_PX = 1.0

# INCLUDE_DEM_ERROR: add the DEM's own vertical error as a third term.
#   False = relative uncertainty (DEM treated as correct) — for comparing
#           features measured against the same DEM.
#   True  = absolute uncertainty — for accuracy against real ground truth.
INCLUDE_DEM_ERROR    = False
DEM_VERTICAL_ERROR_M = 10.0   # vertical RMSE from the DEM product documentation
                              # (ASTER GDEM ~10-15 m, SRTM ~9 m, Copernicus ~4 m)

# ── FOV uncertainty: re-solve the camera at FOV +/- fov_sigma_deg (from Cell 4) ─
# None if the FOV is treated as exact (fov_sigma_deg = 0).
# raycast_scene lets it report any camera that ends up below the terrain surface.
fov_cams = mf.fov_perturbed_cameras(fov, fov_sigma_deg, GCP_PIXEL, GCP_WORLD,
                                    w_img, h_img, RANSAC_THRESHOLD, raycast_scene=scene)

# ── Calculate uncertainty for every feature ───────────────────────────────────
uncertainty_results, REPROJ_PX = mf.line_uncertainty(
    features_world, cam_world, R, K, scene, err, focal_px,
    digitise_px=DIGITISE_PX, include_dem_error=INCLUDE_DEM_ERROR,
    dem_vertical_error_m=DEM_VERTICAL_ERROR_M, camera_lifted=camera_lifted, dZ=dZ,
    fov_cams=fov_cams, fov_sigma_deg=fov_sigma_deg, inlier_mask=inlier_mask)

# ── Write the numbers into the shapefile ──────────────────────────────────────
mf.add_uncertainty_to_shapefile(OUTPUT_SHP, uncertainty_results, DIGITISE_PX, REPROJ_PX,
                                INCLUDE_DEM_ERROR, DEM_VERTICAL_ERROR_M, fov_sigma_deg)

---

# Part 3 — Orthorectification (optional)

Builds an orthorectified GeoTIFF of the photo, and optionally fills unstable
terrain by warping.

**Requires Cells 1–9 to have run in this kernel session.** Part 2 is not needed.

## Cell 14 (Optional) — Orthorectify the Photo

Builds the orthorectified GeoTIFF. Skip this cell if you only want line features
(those do not depend on it).

For every pixel that hit the terrain in Cell 9:

1. Walk along its ray by the recorded distance to get a world position (E, N, Z).
2. Convert that world position to a cell in the output raster.
3. Copy the photo's colour into that cell.

**Stable vs unstable terrain.** Geometry can only be right where the DEM still
describes the landscaoe as the photo saw it. If the DEM is modern and the photo is
from 1950, a glacier that has since thinned by 40 m will be placed in the wrong position —
the ray will hit today's surface, not the surface in the photo.

`STABLE_MASK_PATH` (Cell 2) marks the areas where DEM and photo agree (ie. 'stable ground'). Pixels
landing inside it are written here. Pixels landing outside are left as nodata for
the warp fill in Cell 15. With no mask set, everything is treated as stable and
the ortho is complete after this cell.

Note on "stable" terrain: Ideally this is bedrock, but technically any ground that has not changed by
more than the uncertainty of the method itself counts. E.g., large moraines that haven't changed in the 
50 year time gap that you are measuring? Do you have other evidence to show that it hasn't changed?
Active ice, active shorelines and landslide scars are examples that are unsuitable. 

Build the stable terrain mask in GIS software by digitising stable polygons, or invert a glacier
inventory outline. Either a GeoTIFF (non-zero = stable) or a polygon shapefile
in the DEM's CRS will work.

In [ ]:
# ── Load the stable terrain mask onto the output grid ─────────────────────────
stable_mask_raster = mf.load_stable_mask(STABLE_MASK_PATH, dem)

# ── Back-project every usable pixel and build the orthorectified image ────────
# `ortho` keeps the stable/unstable pixel mapping for Cell 15.
ortho_stack, ortho = mf.orthorectify(img_cv, n_bands, exclusion_mask, depth_image, rays_world,
                                     cam_world, w_img, h_img, dem, stable_mask_raster)

# ── Write the GeoTIFF ─────────────────────────────────────────────────────────
mf.write_ortho_geotiff(OUTPUT_PATH, ortho_stack, ortho, dem, NODATA_VALUE,
                       cam_world, fov, focal_px, err, IMAGE_PATH, STABLE_MASK_PATH, inlier_mask)

# ── Preview orthorectification here ───────────────────────────────────────────
mf.plot_ortho(img_rgb, ortho_stack, ortho['coverage_pct'])

## Cell 15 (Optional) — Fill Unstable Terrain by Warping

Fills the nodata left by Cell 14 where the DEM no longer represnents the photo (eg., glacier retreat).
Requires `STABLE_MASK_PATH` to have been set and Cell 14 to have run with it —
otherwise there is nothing to fill and this cell won't work.

We use the stable pixel locations to fit a relationship between pixel and terrain coordinates to then 
extrapolate unstable pixel world positions. Pixel-world coordinates for 'stable' ground delineated using the
stable ground mask become the 'control points'. Control points are sampled randomly within a grid to ensure
spread across the image and avoid clustering.

**Configuration:** `WARP_METHOD`, `WARP_POLY_ORDER`, `WARP_MAX_CONTROL_PX`,
`OUTPUT_WARP_PATH`

**TPS** (thin plate spline) passes exactly through every control point and handles
awkward control point distributions — bedrock down only two sides of a glacier,
for instance. **Polynomial** is more rigid and therefore more predictable a long
way from any control point. Order 2 is the sensible choice, since order 1 cannot
represent oblique geometry and order 3 tends to swing wildly at the edges.

These filled pixels are an **interpolation and not a measurement**. They are the best
guess at where the photo's colour belongs given the surrounding stable ground, and
they don't carry the accuracy of the ray-traced pixels. **Not ideal to digitise features**.

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
# WARP_METHOD: 'tps' (thin plate spline, recommended) or 'polynomial'.
WARP_METHOD = 'tps'

# WARP_POLY_ORDER: only used when WARP_METHOD = 'polynomial'.
#   1 = affine, 2 = quadratic (recommended), 3 = cubic
WARP_POLY_ORDER = 2

# WARP_MAX_CONTROL_PX: cap on TPS control points. Fitting is O(n^3), so a few
# thousand well-spread points is far better than every stable pixel.
WARP_MAX_CONTROL_PX = 5000                      # doubling this makes the fit about eight times slower

# OUTPUT_WARP_PATH: where the filled ortho goes.
# None = the Cell 14 output path with '_warp' appended.
OUTPUT_WARP_PATH = None

# ── Fit the warp and fill the unstable terrain ────────────────────────────────
# Does nothing (and says why) if Cell 14 has not run or no stable mask was used.
warp = mf.warp_fill(globals().get('ortho'), OUTPUT_PATH, img_cv, w_img, h_img, dem, NODATA_VALUE,
                    WARP_METHOD, WARP_POLY_ORDER, WARP_MAX_CONTROL_PX, OUTPUT_WARP_PATH)

# ── VISUAL CHECK: Before / after / what-was-filled ────────────────────────────
if warp is not None:
    mf.plot_warp_fill(warp)